<h1 style="color:#c8a2c8; font-weight:bold; margin-bottom: 0px; padding-bottom: 0px;">ImmoEliza Project</h1>  
<h2 style="color:#c8a2c8; margin-top: 0px; padding-top: 0px;">ML content</h2>

<h3 style="color:#c8a2c8; font-weight:bold; margin-bottom: 0px; padding-bottom: 0px;">5 -  Model Optimization</h3>  
<u><h3 style="color:#c8a2c8; margin-top: 0px; padding-top: 0px; margin-bottom: 0px; padding-bottom: 0px;">Hyperparameter Tuning</h3></u>

The testing set shall be the LAST thing to evaluate on, and it shall remain unseen until the very end.

In this section I am going to **split training again** to get a **validation set**, which I will use to choose the best hyper-parameters.  

The term for the set of data that is used last is the testing set &rarr; training-validation-test split.  
A rule of thumb is for this split to be 50-30-20, but this can vary according to the quantity of data.  

**Note on Industrial Practice**: While a manual train-validation-test split (e.g., 50-30-20) is a great conceptual approach, it permanently reduces the volume of data available for the model to learn from. In this notebook, we will instead leverage <code>GridSearchCV</code> with <code>cv=5</code> (5-fold Cross-Validation). This technique automates the validation process natively: it keeps the Test Set strictly untouched under lock and key, while dynamically partitioning the Training Set into 5 rotating folds (using 4 for training and 1 for validation at each iteration). This maximizes our training data footprint without any risk of data leakage.

In [2]:
# imports setup

import sys
import os


current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(".."))
utils_path = os.path.join(project_root, "4_Model_Comparison")

if utils_path not in sys.path:
    sys.path.insert(0, utils_path)

from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, train_test_split
import numpy as np
import pandas as pd
import dataframe_image as dfi
import matplotlib.pyplot as plt

# model
from xgboost import XGBRegressor

# metrics
from metrics_utils import calculate_metrics # type: ignore

import warnings
warnings.filterwarnings('ignore')

print("✅ Libraries setup complete.")

✅ Libraries setup complete.


In [3]:
# file location dygnosis
print(f"⚠️ Python is reading from here:\n{os.getcwd()}.")

⚠️ Python is reading from here:
d:\Irene\Desktop\AI_&_Data_Science_training_BeCode\BeCode Projects\Preparation-for-Sprint-3-IMMO-ELIZA-Project\5_Optimization.


In [4]:
# importing features train and test preprocessed datasets

X_TRAIN_CLEAN_CSV_PATH = "../data/cleaned/X_train_clean.csv"
PROP_TEST_PREPROC_CSV_PATH = "../data/cleaned/properties_test_preprocessed.csv"
Y_TRAIN_CLEAN_CSV_PATH = "../data/cleaned/y_train_clean.csv"
Y_TEST_CSV_PATH = "../data/raw/train_test/y_test.csv"

# loading of Train
feature_train = pd.read_csv(X_TRAIN_CLEAN_CSV_PATH)
feature_test = pd.read_csv(PROP_TEST_PREPROC_CSV_PATH)

# loading of Test - to not be touched utill the very end
target_train = pd.read_csv(Y_TRAIN_CLEAN_CSV_PATH)
target_test = pd.read_csv(Y_TEST_CSV_PATH)

print("✅ Preprocessed dataset for Train 'properties_train' successfully loaded.")
print("✅🔒 Preprocessed dataset for Test 'properties_test' successfully loaded.")
print("✅ Target dataset for Train 'target_train' successfully loaded.")
print("✅🔒 Target dataset for Test 'target_test' successfully loaded.")

✅ Preprocessed dataset for Train 'properties_train' successfully loaded.
✅🔒 Preprocessed dataset for Test 'properties_test' successfully loaded.
✅ Target dataset for Train 'target_train' successfully loaded.
✅🔒 Target dataset for Test 'target_test' successfully loaded.


<h3 style="color:#c8a2c8; margin-bottom: 0px; padding-bottom: 0px;">5.1 - 
Starting Baseline Definition</h3>

In [5]:
xgb_opt_results = {}

In [6]:
def xgb_reg_classifier(learning_rate):

    xgb_reg = XGBRegressor(
    n_estimators=100, 
    learning_rate=learning_rate, 
    random_state=42,
    verbosity=1
    )

    return xgb_reg

In [7]:
def xgb_reg(stage, learning_rate, X_train, y_train):
    # invoking xgb_reg_classifier function
    xgb = xgb_reg_classifier(learning_rate)

    # training
    xgb.fit(X_train, y_train) 

    # prediction
    y_prediction_train = xgb.predict(X_train)

    xgb_opt_results[f'XGBoost - {stage}'] = calculate_metrics(y_train, y_prediction_train)

    return xgb_opt_results

In [8]:
winning_lr = 0.1
print(xgb_reg('Start', winning_lr, feature_train, target_train))

{'XGBoost - Start': {'R²': '90.79 %', 'MAE': '53,309.76 €', 'RMSE': '77,509.65 €'}}


<h3 style="color:#c8a2c8; margin-bottom: 0px; padding-bottom: 0px;">5.2 - <code>GridSearchCV</code> function</h3>

In [11]:
def single_grid_search(parameters_grid, learning_rate, X_train, y_train):
    '''
    Performs a grid search using training set given.
    '''
    gridsearch = GridSearchCV(estimator=xgb_reg_classifier(learning_rate),
                              param_grid=parameters_grid, # all parameters I want to test
                              scoring='r2',
                              cv=5, # 5 folds
                              verbose=1,
                              n_jobs=-1
                              )
    return gridsearch.fit(X_train, y_train)

In [12]:
# all parameters I want to test
# first attempt
parameters_grid_1 = {
'max_depth': [3, 4, 5, 6, 7, 8],
'min_child_weight': [1, 3, 5],
'learning_rate': [0.05, 0.06, 0.07, 0.08, 0.09, 0.1],
'n_estimators': [100, 200]
}

In [ ]:
print("Validation ongoing...")
result = single_grid_search(parameters_grid_1, winning_lr, feature_train, target_train)

print(result)
print("Validation completed.")

Fitting 5 folds for each of 216 candidates, totalling 1080 fits
Validation ongoing...
GridSearchCV(cv=5,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=True, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=None, grow_policy=None,
                                    importance_type=None,
                                    interaction_constraints=None,...
                                    max_cat_to_onehot=None, max_delta_step=None,
                                    max_depth=None, max_leaves=None,
                                    min_child_weight=None

Querying the optimizer

In [14]:
# best parameters
print("Best parameters found during validation:")
print(result.best_params_)

# mean score with best parameters
print(f"Mean R² obtained by best parameters found: {result.best_score_:.4f}")

# extracting final pre-trained model object
best_model = result.best_estimator_
print(best_model)

Best parameters found during validation:
{'learning_rate': 0.08, 'max_depth': 7, 'min_child_weight': 5, 'n_estimators': 200}
Mean R² obtained by best parameters found: 0.7663
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.08, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=7,
             max_leaves=None, min_child_weight=5, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=200,
             n_jobs=None, num_parallel_tree=None, ...)


**Before removing outliers** 
- first attempt: returned highest learning_rate (0.1) and highest N_estimators (200) &rarr; **should try higher values**  
best max_depth=4 (max was 8) &rarr; model has still chosen less deep structure to avoid overfitting.  

**After having removed outliers**  
- first attempt, **R² = 76.63%** : returned learning_rate=0.08 and highest N_estimators (200) &rarr; **should try higher n_estimators and higher min_child_weight**  
best max_depth=7 (max was 8)

In [15]:
# all parameters I want to test
# second attempt
parameters_grid_2 = {
'max_depth': [5, 6, 7, 8], # narrow range around the best value found
'min_child_weight': [3, 4, 5, 6, 7], # model seemed to be wanting to increment child_weight -> narrow range around the best value found
'learning_rate': [0.06, 0.07, 0.08, 0.09, 0.10],
'n_estimators': [200, 300, 400] # model seemed to be wanting to increment number of trees, so more trees
}

In [16]:
print("Validation ongoing...")
result = single_grid_search(parameters_grid_2, winning_lr, feature_train, target_train)

print(result)
print("Validation completed.")

Validation ongoing...
Fitting 5 folds for each of 300 candidates, totalling 1500 fits
GridSearchCV(cv=5,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=True, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=None, grow_policy=None,
                                    importance_type=None,
                                    interaction_constraints=None,...
                                    max_cat_to_onehot=None, max_delta_step=None,
                                    max_depth=None, max_leaves=None,
                                    min_child_weight=None

Querying the optimizer

In [17]:
# best parameters
print("Best parameters found during validation:")
print(result.best_params_)

# mean score with best parameters
print(f"Mean R² obtained by best parameters found: {result.best_score_:.4f}")

# extracting final pre-trained model object
best_model = result.best_estimator_
print(best_model)

Best parameters found during validation:
{'learning_rate': 0.07, 'max_depth': 5, 'min_child_weight': 6, 'n_estimators': 400}
Mean R² obtained by best parameters found: 0.7700
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.07, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=6, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=400,
             n_jobs=None, num_parallel_tree=None, ...)


**Before removing outliers**  
- second attempt: XGBoost has extracted virtually all the mathematically possible predictive signal from the current features &rarr; **R² = 66,34%**  
I have reached the model's theoretical limit with the current selected data.  

Time for going back to data and check if there are any features that are currently affecting negatively the score.  



**After having removed outliers**  
- second attempt, **R² = 77.00%** : learning_rate dropped to 0.07 compared to before, max_depth dropped to 5, min_child_weight increased to 6, n_estimators increased to max value (400)


In [18]:
# all parameters I want to test
# second attempt
parameters_grid_3 = {
'max_depth': [4, 5, 6], # narrow range around the best value found
'min_child_weight': [5, 6, 7], # model seemed to be wanting to increment child_weight -> narrow range around the best value found
'learning_rate': [0.06, 0.07, 0.08],
'n_estimators': [300, 400, 500, 600] # model seemed to be wanting to increment number of trees, so more trees
}

In [19]:
print("Validation ongoing...")
result = single_grid_search(parameters_grid_3, winning_lr, feature_train, target_train)

print(result)
print("Validation completed.")

Validation ongoing...
Fitting 5 folds for each of 108 candidates, totalling 540 fits
GridSearchCV(cv=5,
             estimator=XGBRegressor(base_score=None, booster=None,
                                    callbacks=None, colsample_bylevel=None,
                                    colsample_bynode=None,
                                    colsample_bytree=None, device=None,
                                    early_stopping_rounds=None,
                                    enable_categorical=True, eval_metric=None,
                                    feature_types=None, feature_weights=None,
                                    gamma=None, grow_policy=None,
                                    importance_type=None,
                                    interaction_constraints=None,...
                                    max_cat_to_onehot=None, max_delta_step=None,
                                    max_depth=None, max_leaves=None,
                                    min_child_weight=None,

Querying the optimizer

In [20]:
# best parameters
print("Best parameters found during validation:")
print(result.best_params_)

# mean score with best parameters
print(f"Mean R² obtained by best parameters found: {result.best_score_:.4f}")

# extracting final pre-trained model object
best_model = result.best_estimator_
print(best_model)

Best parameters found during validation:
{'learning_rate': 0.07, 'max_depth': 5, 'min_child_weight': 6, 'n_estimators': 600}
Mean R² obtained by best parameters found: 0.7714
XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=True, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.07, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=6, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=600,
             n_jobs=None, num_parallel_tree=None, ...)


**After having removed outliers**  
- third attempt, **R² = 77.14%** : learning_rate still 0.07, max_depth still 5, min_child_weight still 6, n_estimators increased to max value (600)
